OHASE 2: DATA CLEANING FOR FURTHER EDA Analisis


In [10]:
# Helper functions
import pandas as pd
import numpy as np
from pathlib import Path

def missing_info(df):
    missing_info = pd.DataFrame({
        'Missing Count': df.isna().sum(),
        'Percent': (df.isna().mean() * 100).round(2),
        'Data type': df.dtypes,
    })
    
    missing_info = missing_info[missing_info['Missing Count'] > 0]
    return missing_info.sort_values('Percent',ascending=False)
 
def is_duplicates(df):
    print(f'Total Duplicates: {df.duplicated().sum()}')

def dedup(df: pd.DataFrame) -> pd.DataFrame: # should be done 1st
    """
    Remove duplicate property-room records based on (id, occupancy).
    The first occurrence is retained.

        - Duplicate (id, occupancy) pairs
    """

    df = df.drop_duplicates(subset=['id', 'occupancy'], keep='first')
    return df

def drop_columns(df: pd.DataFrame, columns: list[str] | None = None) -> pd.DataFrame:
    """
    Drop columns that are not useful for modeling.

    If columns is not provided, the default set of columns
    identified during EDA will be removed.

    col = ['id', 'title', 'address', 'gate_closing_time', 'total_bathroom']
    """

    drop_cols = ['id', 'title', 'address', 'total_bathrooms', 'warden', 'cooking_allowed', 'gate_closing_time', 'guardian_required', 'nonveg_allowed', 'smoking_allowed']
    if columns is None:
        columns = drop_cols

    df = df.drop(columns=columns)
    return df

def drop_rows(df: pd.DataFrame) -> pd.DataFrame:
    """
    Drop rows that are not useful for modelling.
    
    this func() drops the reows which,
    - ~1% rows with missing values in key columns
    - rent with 0 or NaNs
    - occupancy is NaN
    """

    df = df.dropna(subset=['rent', 'deposit', 'occupancy', 'attached_bathroom'])
    # Remove invalid/placeholder rents.
    # The minimum realistic PG rent in Chennai is well above 1000.
    df = df[df['rent'] >= 1000]

    return df

def fill_boolean_amenities(df: pd.DataFrame) -> pd.DataFrame:
    """
    FIll missing boolean amenites with False
    then convert the columns to bool type
    """
    bool_cols = ['attached_bathroom', 'mess', 'wifi', 'laundry', 'power_backup',
        'refrigerator', 'common_tv', 'room_cleaning','room_ac', 
        'room_cupboard', 'room_tv', 'room_geyser', 'room_bedding',
        'room_attached_bath',
    ]   

    # imputation for amenities
    df[bool_cols] = df[bool_cols].fillna(False).astype(bool)
    # Impute Parking Nan -> 'none'
    df['parking'] = df['parking'].fillna('none')

    assert df[bool_cols].isna().sum().sum() == 0 # should be 0
    assert (df[bool_cols].dtypes == bool).all() # should show: bool

    return df

def impute_transit_score(df: pd.DataFrame) -> pd.DataFrame:
    # this is for fix the left influend skew-ness (should be done before imputation)
    df['transit_score'] = df['transit_score'].replace(-10, np.nan)

    # creating tag for msiing values rows
    df['transit_score_missing'] = df['transit_score'].isna().astype(int)

    # impute with local (locality) median
    df['transit_score'] = (
         
         df.groupby('locality')['transit_score']
        .transform(lambda x: x.fillna(x.median()))
    )

    # impute with global median (if local median is Nan)
    df['transit_score'] = df['transit_score'].fillna(df['transit_score'].median())

    return df

def impute_lifestyle_score(df: pd.DataFrame) -> pd.DataFrame:
    
   # imputation for lifestyle_score
    df['lifestyle_score_missing'] = df['lifestyle_score'].isna().astype(int)  # creating tag for msiing values rows

    df['lifestyle_score'] = ( 
            df.groupby('locality')['lifestyle_score']
            .transform(lambda x: x.fillna(x.median()))
    )  # impute with local (locality) median

    # impute with global median (if local median is Nan)
    df['lifestyle_score'] = df['lifestyle_score'].fillna(df['lifestyle_score'].median()) # impute with global median (if local median is Nan)

    return df

def save_csv(df: pd.DataFrame):
    output_dir = Path('../Data/processed/EDA')
    output_dir.mkdir(parents=True, exist_ok=True)

    # save cleaned dataset
    df.to_csv(output_dir / '01_eda_Phase-2_processed.csv', index=False)
    print("Dataset saved successfully!")

## 2.1 Drop Rows &Columns

In [11]:
df = pd.read_csv(r'D:\chennai1-pg-price-predictor\Data\raw\chennai_pg_dataset.csv')

In [12]:
missing_info(df)

,Missing Count,Percent,Data type
gate_closing_time,1447,87.12,object
common_tv,829,49.91,object
mess,829,49.91,object
power_backup,829,49.91,object
refrigerator,829,49.91,object
wifi,829,49.91,object
cooking_allowed,829,49.91,object
warden,814,49.01,object
laundry,814,49.01,object
room_cleaning,814,49.01,object


In [13]:
df = dedup(df)
df = drop_columns(df)
df = drop_rows(df)
df = fill_boolean_amenities(df)
df = impute_transit_score(df)
df = impute_lifestyle_score(df)
save_csv(df)

C:\Users\phari\AppData\Local\Temp\ipykernel_14916\1667129664.py:76: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[bool_cols] = df[bool_cols].fillna(False).astype(bool)
d:\chennai1-pg-price-predictor\.venv\lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
d:\chennai1-pg-price-predictor\.venv\lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
d:\chennai1-pg-price-predictor\.venv\lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
d:\chennai1-pg-price-predictor\.venv\lib\site-packa

Dataset saved successfully!


In [14]:
is_duplicates(df)

Total Duplicates: 0


In [15]:
missing_info(df)
df['parking'].unique()

array(['Bike', 'Bike and Car', 'none', 'Car'], dtype=object)

In [16]:
df.shape

(1436, 31)

# EDA Phase 2 — Initial Cleaning

## Overview

The purpose of Phase 2 is to apply the data-quality decisions identified in Phase 1 and prepare a cleaner dataset for the remaining EDA stages.

The raw dataset is kept unchanged. This phase focuses only on cleaning and preparation; feature analysis and modelling transformations are not performed here.

### Main cleaning areas

1. Remove duplicate property-room records
2. Remove columns with little or no modelling value
3. Remove rows with invalid or essential missing values
4. Standardize Boolean amenity fields
5. Handle missing transit scores
6. Prepare the cleaned dataset for the next EDA phase


## 1. Duplicate Record Removal

The raw data can contain repeated property listings because the scraping process used overlapping locality definitions.

However, the same property can legitimately have several room configurations such as Single, Double, Three Sharing, or Four Sharing. Therefore, using only the property ID for deduplication would remove valid room variants.

### Cleaning rule

Duplicates are identified using the combination:

`id + occupancy`

Only the first occurrence of each identical pair is retained.

This preserves different occupancy options belonging to the same property.


## 2. Removing Unnecessary Columns

Columns that provide little predictive value, are redundant, or contain very high missingness are removed.

The Phase 2 cleaning removes:

| Column | Reason |
|---|---|
| `id` | Unique identifier rather than a useful predictive feature |
| `title` | Free-text listing name |
| `address` | Detailed address is represented more usefully through locality and coordinates |
| `total_bathrooms` | Building-level aggregate and redundant with room bathroom information |
| `gate_closing_time` | Very high missingness and inconsistent information |
| `warden` | Low variation |
| `cooking_allowed` | Low variation |
| `guardian_required` | Low variation |
| `nonveg_allowed` | Low variation |
| `smoking_allowed` | Low variation |

The purpose is to keep the dataset focused on information that can contribute to later analysis and modelling.


## 3. Removing Invalid Rows

Four fields are treated as important for the cleaned dataset:

- `rent`
- `deposit`
- `occupancy`
- `attached_bathroom`

Rows missing any of these fields are removed.

### Invalid rent values

Rent values below **₹1,000** are treated as unrealistic for the Chennai PG dataset and are removed as likely scraping or data-entry errors.

### Missing occupancy

Rows with missing `occupancy` are removed because occupancy is an important room-level feature and is also part of the duplicate-detection key.

### Cleaning principle

Core information is not artificially imputed when doing so would create an unreliable observation. Instead, incomplete records are removed when the missing field is essential to the analysis.


## 4. Boolean Amenity Cleaning

Fourteen amenity-related columns were stored as `object` values in the raw dataset even though their meaningful values are Boolean.

The affected fields include:

- `attached_bathroom`
- `mess`
- `wifi`
- `laundry`
- `power_backup`
- `refrigerator`
- `common_tv`
- `room_cleaning`
- `room_ac`
- `room_cupboard`
- `room_tv`
- `room_geyser`
- `room_bedding`
- `room_attached_bath`

### Cleaning rule

Missing amenity values are filled with `False`, and the columns are then converted to Boolean dtype.

The interpretation used here is conservative: when the listing does not disclose an amenity, it is treated as unavailable for the modelling dataset.


## 5. Transit Score Imputation

The `transit_score` column contains **-10** as a sentinel value indicating that transit information was unavailable.

It is not treated as a genuine numerical score.

### Steps

1. Replace `-10` with `NaN`.
2. Create `transit_score_missing` to retain information about which rows originally lacked a score.
3. Fill missing values using the median transit score of the same locality.
4. If a locality does not have enough information for a local median, use the overall median as a fallback.

Using the locality median is intended to preserve the local geographic context of the score.


## 6. Lifestyle Score Imputation

The `lifestyle_score` field is also incomplete.

The same locality-based approach is used:

1. Create `lifestyle_score_missing`.
2. Calculate the median lifestyle score within each locality.
3. Use the locality median to fill missing values.
4. Use the overall median as a fallback when necessary.

This keeps the missingness information while producing complete numerical fields for later analysis.


## 7. Parking Preparation

Missing parking values are replaced with:

`none`

This gives the dataset an explicit category for listings where no parking option is recorded, rather than leaving the field as `NaN`.


##  Final Summary

Phase 2 converts the raw dataset into a more consistent analysis-ready dataset.

### Main changes

- Duplicate property/occupancy combinations are removed.
- Identifier, free-text, redundant, low-variance, and highly incomplete columns are dropped.
- Rows missing essential fields are removed.
- Unrealistic rent values below ₹1,000 are removed.
- Amenity columns are converted from object values to Boolean values.
- Missing amenity values are treated as unavailable.
- The `-10` transit sentinel is converted to missing data.
- Locality-level medians are used to fill missing transit and lifestyle scores.
- Missingness indicators are retained for transit and lifestyle scores.
- Missing parking values are represented as `none`.

### Outcome

The result is a cleaner dataset that can be used for the remaining EDA phases without changing the original raw data.
